1️⃣ Architecture Overview
User
  ↓
OpenAI Model (Agent)
  ↓
MCP Client (Python)
  ↓
MCP Server (Tool Provider)
  ↓
MySQL Database

CREATE DATABASE company;

USE company;

CREATE TABLE employees (
    id INT PRIMARY KEY AUTO_INCREMENT,
    name VARCHAR(100),
    department VARCHAR(100),
    salary INT
);

INSERT INTO employees (name, department, salary) VALUES
('Amit Sharma', 'Engineering', 120000),
('Priya Verma', 'Engineering', 95000),
('Rahul Gupta', 'Marketing', 80000),
('Sneha Patil', 'HR', 70000),
('Vikram Singh', 'Engineering', 150000),
('Neha Kulkarni', 'Finance', 110000),
('Arjun Mehta', 'Sales', 90000),
('Pooja Nair', 'Marketing', 85000),
('Rohan Deshmukh', 'Engineering', 105000),
('Kavita Joshi', 'HR', 72000);

Python MCP Tool for MySQL

In [1]:
!pip install openai mysql-connector-python fastapi uvicorn mysql 
!pip install FastMCP

Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sys
import os

print("Python version:", sys.version)
print("Python executable:", sys.executable)

try:
    from mcp.server.fastmcp import FastMCP
    print("✅ FastMCP is now available!")
except ImportError as e:
    print("❌ FastMCP module not found.")
    print("Error:", e)
    print("\nTry these steps:")
    print("1. Install MCP: pip install mcp")
    print("2. Restart the kernel or Python environment")
    print("3. Verify installation using: pip show mcp")

Python version: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]
Python executable: D:\python\python.exe
✅ FastMCP is now available!


import mysql.connector
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("mysql-server")

def run_query(query: str):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="company"
    )

    cursor = conn.cursor(dictionary=True)
    cursor.execute(query)

    result = cursor.fetchall()

    cursor.close()
    conn.close()

    return result


@mcp.tool()
def query_mysql(query: str):
    """Execute a SQL query on MySQL database"""
    return run_query(query)


if __name__ == "__main__":
    mcp.run()

Python Client Using OpenAI + MCP

In [3]:
# Cell 2: Create the server code (weather.py)
# We'll write the server file to disk so the subprocess can run it. host="127.0.0.1", port=8000"
import os
port = int(os.getenv("MCP_PORT", 9000))

server_code = """
from fastmcp import FastMCP
import mysql.connector

app = FastMCP("mysqlserver")

def run_query(query: str):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="company"
    )
    cursor = conn.cursor(dictionary=True)
    cursor.execute(query)
    result = cursor.fetchall()
    cursor.close()
    conn.close()
    return result

@app.tool()
def query_mysql(query: str):
    return run_query(query)

if __name__ == "__main__":
    # For stdio transport (works with stdio_client)
    app.run(transport="sse", host="localhost", port=9000)


    # If you want SSE transport, remove the line above and instead run:
    # app.run(transport="sse")
    # Then start with: uvicorn employeeserver:app --host 127.0.0.1 --port 8000
"""

with open("employeeserver.py", "w") as f:
    f.write(server_code)

print("employeeserver.py written")

employeeserver.py written


In [4]:
!python employeeserver.py



+-----------------------------------------------------------------------------+
|                                                                             |
|                                                                             |
|                        \u2584\u2580\u2580 \u2584\u2580\u2588 \u2588\u2580\u2580 \u2580\u2588\u2580 \u2588\u2580\u2584\u2580\u2588 \u2588\u2580\u2580 \u2588\u2580\u2588                        |
|                        \u2588\u2580  \u2588\u2580\u2588 \u2584\u2584\u2588  \u2588  \u2588 \u2580 \u2588 \u2588\u2584\u2584 \u2588\u2580\u2580                        |
|                                                                             |
|                                                                             |
|                                FastMCP 3.1.1                                |
|                            https://gofastmcp.com                            |
|                                                                        

In [5]:
from mcp.client.sse import sse_client
from mcp.client.session import ClientSession

async def main():
    async with sse_client("http://127.0.0.1:8000/sse") as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Tools:", tools)

            result = await session.call_tool("query_mysql", {"query": "SELECT * FROM employees"})
            print("Result:", result)

await main()

Tools: meta=None nextCursor=None tools=[Tool(name='query_mysql', title=None, description=None, inputSchema={'additionalProperties': False, 'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}, execution=None)]
Result: meta=None content=[TextContent(type='text', text='[{"id":1,"name":"Amit Sharma","department":"Engineering","salary":120000},{"id":2,"name":"Priya Verma","department":"Engineering","salary":95000},{"id":3,"name":"Rahul Gupta","department":"Marketing","salary":80000},{"id":4,"name":"Sneha Patil","department":"HR","salary":70000},{"id":5,"name":"Vikram Singh","department":"Engineering","salary":150000},{"id":6,"name":"Neha Kulkarni","department":"Finance","salary":110000},{"id":7,"name":"Arjun Mehta","department":"Sales","salary":90000},{"id":8,"name":"Pooja Nair","department":"Marketing","salary":85000},{"id":9,"name":"Rohan Deshmukh","department":"Engineering",